**Note:** this notebook was run against sentence/article files on the original author's local machine (see hardcoded `C:\Users\...` paths below) and is kept here as a record of the VADER sentiment aggregation step. Update the input/output paths for your own environment before rerunning.

In [ ]:
import os, sys

# Make sure this notebook works regardless of Jupyter's starting directory:
# CSV paths and the `data_loaders`/`config`/`model` imports below all assume
# the current working directory is the project root (one level up from
# notebooks/).
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), os.pardir))
os.chdir(PROJECT_ROOT)
sys.path.append(PROJECT_ROOT)
print("Project root:", PROJECT_ROOT)


In [4]:
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
import nltk
from tqdm import tqdm

# Download punkt tokenizer if needed
nltk.download('punkt', quiet=True)


def analyze_csv_sentiment_by_sentence(
    csv_path,
    content_column='content',
    output_csv='sentiment_sentence_output.csv',
    batch_size=1000
):
    """
    Reads a CSV of articles, splits each article into sentences,
    runs VADER sentiment on each sentence, and writes a CSV
    with one row per sentence.

    Parameters:
    - csv_path: Path to input CSV file.
    - content_column: Name of the column containing article text.
    - output_csv: Path for the output CSV.
    - batch_size: Number of rows to process before writing to disk.

    Returns: None

    Example usage in a Python script:
        analyze_csv_sentiment_by_sentence(
            'path/to/your.csv',
            content_column='content',
            output_csv='output.csv',
            batch_size=500
        )

    In a Jupyter notebook, just call the function directly after import.
    """
    analyzer = SentimentIntensityAnalyzer()
    reader = pd.read_csv(csv_path, chunksize=batch_size)
    first_write = True

    for chunk in reader:
        records = []
        for _, row in tqdm(chunk.iterrows(), total=len(chunk), desc='Processing articles'):
            article_text = str(row.get(content_column, '')).strip()
            if not article_text:
                continue  # skip empty articles

            # Tokenize into sentences
            sentences = nltk.tokenize.sent_tokenize(article_text)

            for sent_num, sentence in enumerate(sentences, start=1):
                scores = analyzer.polarity_scores(sentence)
                record = {
                    **{col: row[col] for col in chunk.columns if col != content_column},
                    'sentence_number': sent_num,
                    'sentence': sentence,
                    **scores
                }
                records.append(record)

        if records:
            result_df = pd.DataFrame.from_records(records)
            # Write header only on first batch
            result_df.to_csv(
                output_csv,
                mode='w' if first_write else 'a',
                header=first_write,
                index=False
            )
            first_write = False

    print(f"Saved sentence-level sentiment to {output_csv}")



[nltk_data] Error loading punkt: <urlopen error [Errno 11001]
[nltk_data]     getaddrinfo failed>


In [7]:
if __name__ == "__main__":
    analyze_csv_sentiment_by_sentence(
        # Use a raw string or double backslashes for Windows paths:
        r'C:\Users\hardm\OneDrive\Documents\Python Scripts\Masters Project\Data\Tariff_articles.csv',
        content_column='content',
        output_csv='sentiment_sentence_output.csv',
        batch_size=1000
    )

Processing articles: 100%|██████████| 749/749 [00:02<00:00, 274.18it/s]


Saved sentence-level sentiment to sentiment_sentence_output.csv


In [8]:
def aggregate_article_sentiment(
    sentence_csv_path,
    group_by_cols=None,
    output_csv='article_sentiment_output.csv'
):
    """
    Reads a CSV with sentence-level sentiment scores, aggregates the
    sentiment metrics by article, and writes a CSV with one row per article.

    Parameters:
    - sentence_csv_path: Path to the sentence-level CSV output.
    - group_by_cols: List of columns to group by (article identifiers).
        If None, defaults to all columns except 'sentence','sentence_number',
        'neg','neu','pos','compound'.
    - output_csv: Path for the article-level output CSV.

    Returns: None

    Example:
        aggregate_article_sentiment(
            'sentence_sentiment.csv',
            group_by_cols=['url','title','publishedAt','date'],
            output_csv='article_sentiment.csv'
        )
    """
    df = pd.read_csv(sentence_csv_path)
    if group_by_cols is None:
        exclude = {'sentence', 'sentence_number', 'neg', 'neu', 'pos', 'compound'}
        group_by_cols = [col for col in df.columns if col not in exclude]

    # Aggregate mean sentiment per article with custom ordering
    agg_df = (
        df
        .groupby(group_by_cols)
        .agg(
            mean_positive=('pos', 'mean'),
            mean_neutral=('neu', 'mean'),
            mean_negative=('neg', 'mean')
        )
        .reset_index()
    )

    # Ensure column order matches: group_by columns then means
    ordered_cols = group_by_cols + ['mean_positive', 'mean_neutral', 'mean_negative']
    agg_df = agg_df[ordered_cols]

    agg_df.to_csv(output_csv, index=False)
    print(f"Saved article-level sentiment to {output_csv}")

# To run only the aggregation step, use:

if __name__ == '__main__':
    aggregate_article_sentiment(
        sentence_csv_path='sentiment_sentence_output.csv',
        group_by_cols=['url','title','publishedAt','date'],
        output_csv='3D_vadar_Tariff_sentiment.csv'
    )

# To run both steps sequentially, uncomment and adjust paths:
#
# if __name__ == '__main__':
#     analyze_csv_sentiment_by_sentence(
#         'articles.csv',
#         content_column='content',
#         output_csv='sentiment_sentence_output.csv',
#         batch_size=1000
#     )
#     aggregate_article_sentiment(
#         sentence_csv_path='sentiment_sentence_output.csv',
#         group_by_cols=['url','title','publishedAt','date'],
#         output_csv='article_sentiment.csv'
#     )


Saved article-level sentiment to 3D_vadar_Tariff_sentiment.csv
